# 외국인 유학생 건강보험 RAG 챗봇

**대상:** 부산외국어대학교 외국인 유학생  
**지식 소스:** `Health_Insurance_info.pdf` 한 개  
**응답 모델:** `gpt-5.6-luna`  
**검색:** OpenAI 임베딩 + FAISS

이 노트북은 PDF 로딩, 페이지·언어 메타데이터 청킹, 벡터 검색, 엄격한
문서 근거 프롬프트, 한국어·영어 테스트, Streamlit 전체 소스 순서로 구성된다.
문서에 없는 정보는 일반 지식으로 보충하지 않는다.

## 1. 환경 설정

필요 패키지는 `requirements.txt`로 설치한다. API 키는 출력하거나 노트북에 직접
입력하지 않고 `.env`의 `OPENAI_API_KEY` 또는 배포 환경의 Streamlit Secrets에서 읽는다.

```bash
pip install -r requirements.txt
```


In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv

from rag_core import (
    PDF_NAME, DEFAULT_CHAT_MODEL, DEFAULT_EMBEDDING_MODEL,
    load_pdf_pages, build_chunks, create_openai_client,
    embed_texts, create_faiss_index, retrieve,
    build_rag_prompt, answer_question, detect_language,
)

load_dotenv(".env")
ROOT = Path.cwd()
PDF_PATH = ROOT / PDF_NAME
CHAT_MODEL = os.getenv("OPENAI_MODEL", DEFAULT_CHAT_MODEL)
print("PDF:", PDF_PATH.name)
print("Chat model:", CHAT_MODEL)
print("API key configured:", bool(os.getenv("OPENAI_API_KEY") or os.getenv("model_api_key")))

PDF: Health_Insurance_info.pdf
Chat model: gpt-5.6-luna
API key configured: True


## 2. PDF 로딩과 텍스트 확인

PyMuPDF로 7개 페이지를 읽는다. 페이지 번호가 1~3이면 한국어(`ko`),
4~7이면 영어(`en`) 메타데이터를 부여한다.

In [2]:
pages = load_pdf_pages(PDF_PATH)
print("페이지 수:", len(pages))
for page in pages:
    preview = str(page["text"]).replace("\n", " ")[:150]
    print(f'p.{page["page"]} | {page["language"]} | {len(page["text"]):,}자 | {preview}...')

페이지 수: 7
p.1 | ko | 1,213자 | 외국인 유학생 보험 안내 1. 가입 대상자: 외국인등록을 마친 외국인 학생(D-2 비자 소지자) 및 재외국민 등 1) 2021년 3월 1일부터 외국인등록을 마친 외국인 학생(D-2 소지자)은 국민건강보험에 자동가입 됩니다. (비자 유형에 따라 가입일이 다를 수 있음) ...
p.2 | ko | 1,291자 | ※ 학교선택은 Busan University of Foreign Studies 선택 ⋅신청이 완료되면 보험사에서 이메일, 카톡 등으로 보험료 납부 안내 등 개별연락 나. 국민건강보험 (NHIS) 1. 보험료 (지역가입자 기준) - 2023년 3월~ : 평균 보험료의 5...
p.3 | ko | 677자 | 5. 국민건강보험 자격득실확인서: 보험에 가입되었음을 확인할 수 있는 증명서 6. 국민건강보험공단 금정지사 부산외국어대학교 국제학생지원팀 유형 방법 팩스/ 이메일 1) 국민건강보험공단(T. 033-811-2000) 2) 언어 선택(① 영어/한국어 ② 중국어 ③ 베트남어...
p.4 | en | 2,196자 | Insurance Guide for BUFS International Students 1. Eligible Applicants: Students who have completed Alien Registration ※ RC registration may take time...
p.5 | en | 2,337자 | 2) Insurance Period and Premium - Since it takes 1-2 months to enroll in National Health Insurance, it is recommended to have private insurance for at...
p.6 | en | 1,980자 | (IMPORTANT!!) Your examination at a tertiary hospital may be rejected, or you may not be able to use yo

## 3. 청킹 및 메타데이터 구성

청크는 페이지 경계를 넘지 않게 생성한다. 각 청크는 `source`, `page`,
`language`, `chunk_id`를 가지므로 검색 결과를 정확한 PDF 페이지로 표시할 수 있다.

In [3]:
chunks = build_chunks(pages, max_chars=900, overlap=140)
print("청크 수:", len(chunks))
for chunk in chunks:
    print(chunk.chunk_id, "| page", chunk.page, "|", chunk.language, "|", len(chunk.text), "chars")

청크 수: 15
p1-1-dce503fe5175 | page 1 | ko | 899 chars
p1-2-7bfdbc147da6 | page 1 | ko | 313 chars
p2-1-2df1d8b86c67 | page 2 | ko | 899 chars
p2-2-1086a58b53b9 | page 2 | ko | 390 chars
p3-1-f9d2b87d6d89 | page 3 | ko | 677 chars
p4-1-9efb7365e815 | page 4 | en | 896 chars
p4-2-95e813bc8190 | page 4 | en | 836 chars
p4-3-ce246891918b | page 4 | en | 462 chars
p5-1-74bfc804936b | page 5 | en | 881 chars
p5-2-3af5ef825294 | page 5 | en | 896 chars
p5-3-b48081ee0630 | page 5 | en | 558 chars
p6-1-331a3d02e38f | page 6 | en | 890 chars
p6-2-bf008fcbb368 | page 6 | en | 871 chars
p6-3-aec71f8500c3 | page 6 | en | 217 chars
p7-1-c0a306ba6c27 | page 7 | en | 431 chars


## 4. 임베딩과 FAISS 벡터 저장소

`text-embedding-3-small`로 청크 임베딩을 만들고 L2 정규화한 뒤,
FAISS `IndexFlatIP`에 저장하여 코사인 유사도 검색을 수행한다.

In [4]:
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("model_api_key")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in .env before running API cells.")

client = create_openai_client(api_key)
chunk_vectors = embed_texts(
    client,
    [chunk.text for chunk in chunks],
    model=DEFAULT_EMBEDDING_MODEL,
)
index = create_faiss_index(chunk_vectors)
print("FAISS vectors:", index.ntotal, "| dimension:", index.d)

FAISS vectors: 15 | dimension: 1536


## 5. 검색 결과 확인

In [5]:
sample_question = "자격득실확인서는 어떻게 발급받나요?"
sample_language = detect_language(sample_question)
sample_hits = retrieve(
    sample_question, client, index, chunks,
    language=sample_language, top_k=5,
)
for rank, hit in enumerate(sample_hits, 1):
    print(f"{rank}. {hit.chunk.source} p.{hit.chunk.page} | score={hit.score:.3f}")
    print(hit.chunk.text.replace("\n", " ")[:220], "\n")

1. Health_Insurance_info.pdf p.3 | score=0.399
5. 국민건강보험 자격득실확인서: 보험에 가입되었음을 확인할 수 있는 증명서 6. 국민건강보험공단 금정지사 부산외국어대학교 국제학생지원팀 유형 방법 팩스/ 이메일 1) 국민건강보험공단(T. 033-811-2000) 2) 언어 선택(① 영어/한국어 ② 중국어 ③ 베트남어 ④ 우즈베크어) 3) 개인정보 제공(이름, 외국인등록번호) 4) 건강보험자격득실확인서 요청 5) 팩스 번호 또는 이메일 주소 

2. Health_Insurance_info.pdf p.2 | score=0.324
※ 학교선택은 Busan University of Foreign Studies 선택 ⋅신청이 완료되면 보험사에서 이메일, 카톡 등으로 보험료 납부 안내 등 개별연락 나. 국민건강보험 (NHIS) 1. 보험료 (지역가입자 기준) - 2023년 3월~ : 평균 보험료의 50% 감액(약 7만 5천 원/월) ※ 비자 유형에 따라 보험료 상이 (국민건강보험공단 문의) 2. 보험료 납부 방법 1) 외국 

3. Health_Insurance_info.pdf p.2 | score=0.304
에만 해외 체류 동안의 보험료를 모두 납부하면 재가입됩니다. (D-2 소지자 제외, 미납 요금 납부할 필요 없 음) ※ 한국 “재입국”의 경우 자동 재가입이 아니며, 반드시 국민건강보험에 연락해야 함 ※ 해외 체류 동안 보험료를 미납하는 경우, 입국 6개월 후 보험 재가입 가능 ※ “개인정보보호법”으로 인해 국민건강보험 관련 문의는 학교에서 상담이 불가하므로 국민 건강보험공단으로 직접 문의  

4. Health_Insurance_info.pdf p.1 | score=0.277
- 건강보험가입까지 1~2개월 소요되기 때문에 최소 2개월 이상 가입을 권장합니다. - 국내보험료는 2개월 기준 나이에 따라 3~7만원 가량입니다. 3) 보험가입방법(국내) - 학생이 직접 신청 (하단 사이트에서 신청) ⋅www.sos911.co.kr (한

## 6. 프롬프트와 RAG 체인

프롬프트는 다음을 강제한다.

- 제공된 PDF 발췌문만 사용
- 근거가 없으면 확인 불가 메시지
- 질문 언어로 답변
- `핵심 답변 → 절차 → 주의사항 → PDF 페이지`
- 변경 가능 정보는 NHIS 재확인
- BUFS/PNU 표기 충돌은 학교명 질문에서 명시하고 임의 수정 금지

In [6]:
prompt = build_rag_prompt(sample_question, sample_hits, sample_language)
print(prompt[:2200])

You are a document-grounded assistant for international students at Busan University of Foreign Studies.

Rules:
1. Use only the excerpts below. Do not add general knowledge, assumptions, or updated facts from memory.
2. If the excerpts do not directly support the answer, use the required unsupported sentence.
3. Cite only these retrieved page numbers: [1, 2, 3].
4. Preserve numbers, dates, phone numbers, addresses, and procedures exactly as written.
5. Do not discuss the BUFS/PNU wording conflict unless the question is about the university or school name.
6. Do not claim that an old premium, policy, contact, or procedure is current. Say that the PDF states it and advise NHIS reconfirmation.
7. Keep the response concise and practical.

반드시 한국어로 다음 형식을 사용하세요.
핵심 답변: 2~4문장
절차:
1. 필요한 순서
주의사항:
- 문서에 적힌 주의사항
PDF 페이지: Health_Insurance_info.pdf p. X[, p. Y]

근거가 없으면 정확히 다음 한 문장만 출력하세요:
제공된 Health_Insurance_info.pdf에서 이 질문에 대한 내용을 확인할 수 없습니다. 국민건강보험공단(033-811-2000)에 직접 확인해 주세요.

Question:
자격득

## 7. 한국어·영어·범위 밖 질문 테스트

In [7]:
test_questions = [
    ("국민건강보험료는 어떻게 납부하나요?", "ko"),
    ("How can I get an NHIS certificate?", "en"),
    ("부산에서 가장 맛있는 식당은 어디인가요?", "ko"),
]

test_results = []
for question, language in test_questions:
    hits = retrieve(question, client, index, chunks, language=language, top_k=5)
    answer = answer_question(question, hits, client, language, model=CHAT_MODEL)
    pages_used = sorted({hit.chunk.page for hit in hits})
    test_results.append((question, language, pages_used, answer))
    print("=" * 90)
    print("Q:", question)
    print("Retrieved pages:", pages_used)
    print("A:", answer)

Q: 국민건강보험료는 어떻게 납부하나요?
Retrieved pages: [1, 2, 3]
A: 핵심 답변: 국민건강보험료는 매달 발송되는 고지서(지로)를 확인한 후, 고지서상의 계좌로 기간 내 납부합니다. 납부 방법은 자동이체/APP, 은행 방문, 인터넷/ATM 중 하나를 선택할 수 있습니다. PDF에는 다음 달 보험료를 매월 25일까지 미리 납부해야 한다고 기재되어 있습니다.

절차:
1. 외국인등록증상의 주소로 발송된 고지서(지로)를 확인합니다.
2. 고지서상의 계좌로 납부합니다.
3. 자동이체/APP은 국민건강보험공단 전화 또는 방문을 통해 신청하고, 은행 방문은 고지서를 들고 영업시간 내 국내 은행을 방문하며, 인터넷/ATM은 고지서상의 가상계좌로 납부합니다.

주의사항:
- 다음 달 보험료를 매월 25일까지 미리 납부해야 하며 미납 시 병원 이용과 체류에 불이익이 있을 수 있습니다.
- 주소가 변경된 학생은 주소 변경신청을 해야 합니다.
- 문서 내용은 변경될 수 있으므로 정확한 납부 방법과 금액은 국민건강보험공단에 재확인하세요. 문의: 033-811-2000

PDF 페이지: Health_Insurance_info.pdf p. 1, p. 2


Q: How can I get an NHIS certificate?
Retrieved pages: [4, 5, 6]
A: Key answer: You can obtain an NHIS Certificate of Qualification for Health Insurance (건강보험자격득실확인서) by calling NHIS or using the NHIS website. The PDF states these procedures; please reconfirm them with NHIS at 033-811-2000.

Steps:
1. Required sequence
   - **By phone:** Call 033-811-2000 → select your language (① ENG/KOR ② CHN ③ VNM ④ UZB) → provide personal information (Name, RC number, etc.) → request a “Certificate of Qualification for Health Insurance (건강보험자격득실확인서)” → provide a personal fax number/e-mail address.
   - **Online:** Go to https://www.nhis.or.kr/nhis/index.do → click “자격득실확인서 발급”(Issue my insurance certificate) → log in as a private/company user. An “Official Certificate(공인인증서)” is required, obtained by visiting your bank in Korea individually.

Important notes:
- NHIS service hours are Mon~Fri, 09:00~18:00 (KST).
- Inquiries related to NHIS must be individually made with NHIS for personal-information

Q: 부산에서 가장 맛있는 식당은 어디인가요?
Retrieved pages: [1, 2, 3]
A: 제공된 Health_Insurance_info.pdf에서 이 질문에 대한 내용을 확인할 수 없습니다. 국민건강보험공단(033-811-2000)에 직접 확인해 주세요.


## 8. 테스트 판정

정상 답변은 질문 언어와 PDF 페이지 표시를 확인한다. 범위 밖 질문은
`Health_Insurance_info.pdf`에서 확인할 수 없다는 메시지를 반환해야 한다.

In [8]:
assert "PDF 페이지:" in test_results[0][3]
assert "PDF pages:" in test_results[1][3]
assert "확인할 수 없습니다" in test_results[2][3]
print("한국어/영어/범위 밖 테스트: PASS")

한국어/영어/범위 밖 테스트: PASS


## 9. Streamlit 앱 전체 코드

아래 두 파일이 배포 앱 전체 구현이다. `app.py`는 UI와 세션 상태를,
`rag_core.py`는 PDF 처리·FAISS 검색·Responses API 호출을 담당한다.

### `rag_core.py`

```python
"""Core RAG utilities for the BUFS international student insurance chatbot."""

from __future__ import annotations

import hashlib
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

import faiss
import fitz
import numpy as np
from openai import OpenAI


PDF_NAME = "Health_Insurance_info.pdf"
DEFAULT_CHAT_MODEL = "gpt-5.6-luna"
DEFAULT_EMBEDDING_MODEL = "text-embedding-3-small"

UNSUPPORTED_KO = (
    "제공된 Health_Insurance_info.pdf에서 이 질문에 대한 내용을 확인할 수 없습니다. "
    "국민건강보험공단(033-811-2000)에 직접 확인해 주세요."
)
UNSUPPORTED_EN = (
    "I cannot verify the answer from Health_Insurance_info.pdf. "
    "Please contact the National Health Insurance Service at 033-811-2000."
)

CHANGE_NOTICE_KO = (
    "보험료와 제도는 변경될 수 있습니다. 최신 내용은 국민건강보험공단 홈페이지 "
    "또는 외국인 전용 번호(033-811-2000)로 재확인하세요."
)
CHANGE_NOTICE_EN = (
    "Premiums and policies may change. Reconfirm the latest information on the NHIS website "
    "or through the foreigners' helpline (033-811-2000)."
)

SCHOOL_CONFLICT_KO = (
    "문서 표기 주의: 영어 4페이지의 제목은 BUFS이지만 본문에는 PNU와 PNU Group "
    "Insurance라는 표현이 남아 있습니다. 챗봇은 이를 임의로 수정하지 않습니다."
)
SCHOOL_CONFLICT_EN = (
    "Document wording note: the English title on page 4 says BUFS, while the body still contains "
    "\"PNU\" and \"PNU Group Insurance.\" The chatbot does not silently correct this conflict."
)


@dataclass(frozen=True)
class Chunk:
    text: str
    page: int
    language: str
    source: str = PDF_NAME
    chunk_id: str = ""


@dataclass(frozen=True)
class SearchHit:
    chunk: Chunk
    score: float


def normalize_pdf_text(text: str) -> str:
    """Repair common extraction artifacts while preserving useful line boundaries."""
    text = text.replace("\u3000", " ").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_pdf_pages(pdf_path: str | Path) -> list[dict[str, object]]:
    """Load the only approved knowledge source with page and language metadata."""
    pdf_path = Path(pdf_path)
    if pdf_path.name != PDF_NAME:
        raise ValueError(f"Only {PDF_NAME} may be used as the knowledge source.")
    if not pdf_path.exists():
        raise FileNotFoundError(f"Missing required PDF: {pdf_path}")

    document = fitz.open(pdf_path)
    pages: list[dict[str, object]] = []
    try:
        for index, page in enumerate(document, start=1):
            text = normalize_pdf_text(page.get_text("text"))
            pages.append(
                {
                    "page": index,
                    "language": "ko" if index <= 3 else "en",
                    "source": PDF_NAME,
                    "text": text,
                }
            )
    finally:
        document.close()
    return pages


def _split_long_block(block: str, max_chars: int, overlap: int) -> Iterable[str]:
    if len(block) <= max_chars:
        yield block
        return
    start = 0
    while start < len(block):
        end = min(start + max_chars, len(block))
        if end < len(block):
            boundary = max(
                block.rfind("\n", start, end),
                block.rfind(". ", start, end),
                block.rfind("다. ", start, end),
            )
            if boundary > start + max_chars // 2:
                end = boundary + 1
        yield block[start:end].strip()
        if end >= len(block):
            break
        start = max(0, end - overlap)


def build_chunks(
    pages: Sequence[dict[str, object]],
    max_chars: int = 900,
    overlap: int = 140,
) -> list[Chunk]:
    """Create page-bounded chunks so every retrieved passage has an exact citation."""
    chunks: list[Chunk] = []
    for page_data in pages:
        text = str(page_data["text"])
        raw_blocks = [part.strip() for part in re.split(r"\n(?=\S)", text) if part.strip()]
        buffer = ""
        page_chunks: list[str] = []
        for block in raw_blocks:
            candidate = f"{buffer}\n{block}".strip() if buffer else block
            if len(candidate) <= max_chars:
                buffer = candidate
                continue
            if buffer:
                page_chunks.extend(_split_long_block(buffer, max_chars, overlap))
            buffer = block
        if buffer:
            page_chunks.extend(_split_long_block(buffer, max_chars, overlap))

        for local_index, chunk_text in enumerate(page_chunks, start=1):
            digest = hashlib.sha1(
                f"{page_data['page']}:{local_index}:{chunk_text}".encode("utf-8")
            ).hexdigest()[:12]
            chunks.append(
                Chunk(
                    text=chunk_text,
                    page=int(page_data["page"]),
                    language=str(page_data["language"]),
                    source=str(page_data["source"]),
                    chunk_id=f"p{page_data['page']}-{local_index}-{digest}",
                )
            )
    return chunks


def create_openai_client(api_key: str) -> OpenAI:
    if not api_key or not api_key.strip():
        raise ValueError("OPENAI_API_KEY is required.")
    return OpenAI(api_key=api_key.strip())


def embed_texts(
    client: OpenAI,
    texts: Sequence[str],
    model: str = DEFAULT_EMBEDDING_MODEL,
) -> np.ndarray:
    response = client.embeddings.create(model=model, input=list(texts))
    vectors = np.asarray([item.embedding for item in response.data], dtype="float32")
    faiss.normalize_L2(vectors)
    return vectors


def create_faiss_index(vectors: np.ndarray) -> faiss.IndexFlatIP:
    if vectors.ndim != 2 or vectors.shape[0] == 0:
        raise ValueError("A non-empty 2D embedding matrix is required.")
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def retrieve(
    query: str,
    client: OpenAI,
    index: faiss.IndexFlatIP,
    chunks: Sequence[Chunk],
    language: str,
    embedding_model: str = DEFAULT_EMBEDDING_MODEL,
    top_k: int = 5,
) -> list[SearchHit]:
    query_vector = embed_texts(client, [query], embedding_model)
    candidate_count = min(len(chunks), max(top_k * 3, top_k))
    scores, indices = index.search(query_vector, candidate_count)
    preferred = [
        SearchHit(chunks[int(idx)], float(score))
        for score, idx in zip(scores[0], indices[0])
        if idx >= 0 and chunks[int(idx)].language == language
    ]
    fallback = [
        SearchHit(chunks[int(idx)], float(score))
        for score, idx in zip(scores[0], indices[0])
        if idx >= 0 and chunks[int(idx)].language != language
    ]
    return (preferred + fallback)[:top_k]


def detect_language(text: str, selected: str = "auto") -> str:
    if selected in {"ko", "en"}:
        return selected
    return "ko" if re.search(r"[가-힣]", text) else "en"


def _context_text(hits: Sequence[SearchHit]) -> str:
    sections = []
    for hit in hits:
        sections.append(
            f"[SOURCE: {hit.chunk.source}, PAGE: {hit.chunk.page}, "
            f"LANGUAGE: {hit.chunk.language}]\n{hit.chunk.text}"
        )
    return "\n\n".join(sections)


def build_rag_prompt(
    question: str,
    hits: Sequence[SearchHit],
    language: str,
) -> str:
    allowed_pages = sorted({hit.chunk.page for hit in hits})
    school_terms = (
        "학교",
        "대학",
        "부산외대",
        "부산외국어대학교",
        "bufs",
        "pnu",
        "university",
        "school",
    )
    conflict_rule = (
        f"5. Explicitly state this wording conflict: {SCHOOL_CONFLICT_KO if language == 'ko' else SCHOOL_CONFLICT_EN}"
        if any(term in question.lower() for term in school_terms)
        else "5. Do not discuss the BUFS/PNU wording conflict unless the question is about the university or school name."
    )
    if language == "ko":
        format_instruction = """반드시 한국어로 다음 형식을 사용하세요.
핵심 답변: 2~4문장
절차:
1. 필요한 순서
주의사항:
- 문서에 적힌 주의사항
PDF 페이지: Health_Insurance_info.pdf p. X[, p. Y]"""
        unsupported = f"근거가 없으면 정확히 다음 한 문장만 출력하세요:\n{UNSUPPORTED_KO}"
    else:
        format_instruction = """Answer in English using exactly this structure.
Key answer: 2-4 sentences
Steps:
1. Required sequence
Important notes:
- Notes stated in the document
PDF pages: Health_Insurance_info.pdf p. X[, p. Y]"""
        unsupported = f"If the evidence is insufficient, output exactly this sentence:\n{UNSUPPORTED_EN}"

    return f"""You are a document-grounded assistant for international students at Busan University of Foreign Studies.

Rules:
1. Use only the excerpts below. Do not add general knowledge, assumptions, or updated facts from memory.
2. If the excerpts do not directly support the answer, use the required unsupported sentence.
3. Cite only these retrieved page numbers: {allowed_pages}.
4. Preserve numbers, dates, phone numbers, addresses, and procedures exactly as written.
{conflict_rule}
6. Do not claim that an old premium, policy, contact, or procedure is current. Say that the PDF states it and advise NHIS reconfirmation.
7. Keep the response concise and practical.

{format_instruction}

{unsupported}

Question:
{question}

Approved excerpts:
{_context_text(hits)}
"""


def answer_question(
    question: str,
    hits: Sequence[SearchHit],
    client: OpenAI,
    language: str,
    model: str = DEFAULT_CHAT_MODEL,
) -> str:
    prompt = build_rag_prompt(question, hits, language)
    response = client.responses.create(
        model=model,
        reasoning={"effort": "low"},
        max_output_tokens=900,
        input=prompt,
    )
    answer = response.output_text.strip()
    if not answer:
        return UNSUPPORTED_KO if language == "ko" else UNSUPPORTED_EN
    return answer


def should_show_change_notice(answer: str, language: str) -> bool:
    if answer in {UNSUPPORTED_KO, UNSUPPORTED_EN}:
        return False
    keywords = (
        ("보험료", "납부", "자격", "가입", "비자", "연락처", "공단", "제도")
        if language == "ko"
        else ("premium", "payment", "eligibility", "enroll", "visa", "contact", "nhis", "policy")
    )
    lowered = answer.lower()
    return any(keyword in lowered for keyword in keywords)


def extract_cited_pages(answer: str, hits: Sequence[SearchHit]) -> list[SearchHit]:
    """Keep UI source cards aligned with page numbers cited in the generated answer."""
    cited = {
        int(value)
        for value in re.findall(r"(?:p\.?|페이지[:\s]*)\s*(\d+)", answer, flags=re.IGNORECASE)
    }
    if not cited:
        return list(hits)
    return [hit for hit in hits if hit.chunk.page in cited]


def model_name() -> str:
    return os.getenv("OPENAI_MODEL", DEFAULT_CHAT_MODEL)

```

### `app.py`

```python
"""Streamlit entry point for the BUFS international student insurance RAG chatbot."""

from __future__ import annotations

import os
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv

from rag_core import (
    CHANGE_NOTICE_EN,
    CHANGE_NOTICE_KO,
    DEFAULT_CHAT_MODEL,
    DEFAULT_EMBEDDING_MODEL,
    PDF_NAME,
    SCHOOL_CONFLICT_EN,
    SCHOOL_CONFLICT_KO,
    UNSUPPORTED_EN,
    UNSUPPORTED_KO,
    answer_question,
    build_chunks,
    create_faiss_index,
    create_openai_client,
    detect_language,
    embed_texts,
    extract_cited_pages,
    load_pdf_pages,
    model_name,
    retrieve,
    should_show_change_notice,
)


load_dotenv(Path(__file__).with_name(".env"))

st.set_page_config(
    page_title="BUFS Health Insurance Guide",
    page_icon="🏥",
    layout="centered",
    initial_sidebar_state="expanded",
)

st.markdown(
    """
    <style>
      :root {
        --kakao-yellow: #FEE500;
        --kakao-ink: #191919;
        --telegram-blue: #229ED9;
        --telegram-deep: #168AC1;
        --chat-bg: #EEF4F8;
        --line: #DCE7EE;
        --muted: #6D7E8A;
      }

      html, body, [class*="css"] {
        font-family: -apple-system, BlinkMacSystemFont, "Apple SD Gothic Neo",
                     "Segoe UI", sans-serif;
      }
      .stApp {
        background:
          radial-gradient(circle at 18% 0%, rgba(34,158,217,.10), transparent 28rem),
          linear-gradient(180deg, #F8FBFD 0%, var(--chat-bg) 100%);
      }
      [data-testid="stHeader"] {background: transparent;}
      .block-container {
        max-width: 900px;
        padding-top: 1.15rem;
        padding-bottom: 7rem;
      }

      [data-testid="stSidebar"] {
        background: linear-gradient(180deg, #FFFFFF 0%, #F4F9FC 100%);
        border-right: 1px solid #DDE9F0;
        box-shadow: 10px 0 30px rgba(50, 88, 110, .05);
      }
      [data-testid="stSidebar"] .block-container {padding-top: 1.3rem;}

      .side-profile {
        display: flex;
        align-items: center;
        gap: .8rem;
        padding: .85rem;
        margin-bottom: 1.1rem;
        border: 1px solid #DCE9F0;
        border-radius: 18px;
        background: #FFFFFF;
        box-shadow: 0 8px 24px rgba(30, 86, 116, .07);
      }
      .side-avatar, .profile-avatar, .welcome-avatar {
        display: grid;
        place-items: center;
        flex: 0 0 auto;
        font-weight: 800;
      }
      .side-avatar {
        width: 42px; height: 42px;
        border-radius: 14px;
        background: var(--kakao-yellow);
        color: var(--kakao-ink);
        font-size: 1.25rem;
      }
      .side-profile strong {display: block; color: #18313F; font-size: .95rem;}
      .side-profile small {color: var(--muted); font-size: .77rem;}

      .messenger-header {
        display: flex;
        align-items: center;
        gap: 1rem;
        min-height: 92px;
        padding: 1.1rem 1.25rem;
        border-radius: 24px 24px 16px 16px;
        background: linear-gradient(135deg, var(--telegram-blue), #54B8E6);
        color: white;
        box-shadow: 0 14px 34px rgba(34, 158, 217, .24);
      }
      .profile-avatar {
        position: relative;
        width: 58px; height: 58px;
        border-radius: 20px;
        background: var(--kakao-yellow);
        color: var(--kakao-ink);
        font-size: 1.55rem;
        box-shadow: inset 0 -3px 0 rgba(0,0,0,.08), 0 8px 18px rgba(0,0,0,.12);
      }
      .online-dot {
        position: absolute;
        right: -2px; bottom: -2px;
        width: 15px; height: 15px;
        border: 3px solid #FFFFFF;
        border-radius: 50%;
        background: #38D67A;
      }
      .profile-copy {min-width: 0; flex: 1;}
      .profile-copy h1 {
        margin: 0 0 .2rem;
        color: white;
        font-size: 1.52rem;
        letter-spacing: -.03em;
      }
      .profile-copy p {
        margin: 0;
        color: rgba(255,255,255,.88);
        font-size: .87rem;
      }
      .header-badge {
        padding: .45rem .7rem;
        border: 1px solid rgba(255,255,255,.35);
        border-radius: 999px;
        background: rgba(255,255,255,.16);
        color: white;
        font-size: .72rem;
        font-weight: 800;
        letter-spacing: .06em;
      }
      .status-strip {
        display: flex;
        flex-wrap: wrap;
        gap: .45rem;
        padding: .75rem .25rem 1.05rem;
      }
      .status-pill {
        display: inline-flex;
        align-items: center;
        gap: .3rem;
        padding: .38rem .62rem;
        border: 1px solid #D8E6ED;
        border-radius: 999px;
        background: rgba(255,255,255,.8);
        color: #49616F;
        font-size: .75rem;
        font-weight: 650;
      }

      .welcome-row {
        display: flex;
        align-items: flex-start;
        gap: .65rem;
        margin: .4rem 0 1.35rem;
      }
      .welcome-avatar {
        width: 38px; height: 38px;
        border-radius: 14px;
        background: var(--telegram-blue);
        color: white;
        font-size: .72rem;
        box-shadow: 0 6px 14px rgba(34,158,217,.2);
      }
      .welcome-bubble {
        max-width: 76%;
        padding: .82rem 1rem;
        border: 1px solid #DCE8EF;
        border-radius: 7px 18px 18px 18px;
        background: white;
        color: #243A47;
        box-shadow: 0 7px 20px rgba(35, 74, 96, .07);
        line-height: 1.55;
        font-size: .92rem;
      }
      .welcome-bubble strong {color: var(--telegram-deep);}
      .welcome-bubble small {display:block; margin-top:.35rem; color:#7B8B95;}

      .quick-label {
        display: flex;
        align-items: center;
        justify-content: space-between;
        margin: .3rem 0 .6rem;
      }
      .quick-label strong {color: #294453; font-size: .96rem;}
      .quick-label span {color: #8798A2; font-size: .74rem;}

      div[data-testid="stButton"] > button {
        min-height: 3.05rem;
        justify-content: flex-start;
        padding: .65rem .85rem;
        border: 1px solid #D7E4EB;
        border-radius: 15px;
        background: rgba(255,255,255,.92);
        color: #29414F;
        font-weight: 700;
        box-shadow: 0 5px 14px rgba(39, 77, 98, .055);
        transition: transform .16s ease, border-color .16s ease, background .16s ease;
      }
      div[data-testid="stButton"] > button:hover {
        transform: translateY(-2px);
        border-color: var(--kakao-yellow);
        background: #FFFDEA;
        color: #111111;
      }
      div[data-testid="stButton"] > button:focus {
        box-shadow: 0 0 0 3px rgba(34,158,217,.16);
      }

      [data-testid="stChatMessage"] {
        gap: .65rem;
        padding: .55rem .2rem;
        background: transparent;
      }
      [data-testid="stChatMessage"] [data-testid="stChatMessageContent"] {
        max-width: 78%;
        padding: .72rem .95rem;
        border: 1px solid #DCE8EF;
        border-radius: 7px 18px 18px 18px;
        background: #FFFFFF;
        box-shadow: 0 7px 20px rgba(35, 74, 96, .07);
      }
      [data-testid="stChatMessage"]:has([data-testid="stChatMessageAvatarUser"]) {
        flex-direction: row-reverse;
      }
      [data-testid="stChatMessage"]:has([data-testid="stChatMessageAvatarUser"]) [data-testid="stChatMessageContent"] {
        border-color: #F0DA00;
        border-radius: 18px 7px 18px 18px;
        background: var(--kakao-yellow);
        color: var(--kakao-ink);
        box-shadow: 0 7px 18px rgba(170, 148, 0, .12);
      }
      [data-testid="stChatMessage"] [data-testid="stChatMessageAvatarUser"] {
        background: var(--kakao-yellow);
        color: #1A1A1A;
      }
      [data-testid="stChatMessage"] [data-testid="stChatMessageAvatarAssistant"] {
        background: var(--telegram-blue);
        color: white;
      }

      [data-testid="stChatInput"] {
        border: 1px solid #CFE0E9;
        border-radius: 999px;
        background: #FFFFFF;
        box-shadow: 0 12px 32px rgba(31, 76, 101, .16);
      }
      [data-testid="stChatInput"] textarea {
        min-height: 52px;
        padding-left: .45rem;
      }
      [data-testid="stChatInput"] button {
        border-radius: 50%;
        background: var(--telegram-blue);
        color: white;
      }

      .source-card {
        border: 1px solid #D9E6ED;
        border-left: 4px solid var(--telegram-blue);
        border-radius: 13px;
        padding: .82rem .9rem;
        margin: .5rem 0;
        background: #F7FBFD;
      }
      .source-card strong {color: #197FB0;}
      .source-card small {color: #526B79; line-height: 1.5;}
      [data-testid="stExpander"] {
        border: 1px solid #DCE8EF;
        border-radius: 13px;
        background: rgba(255,255,255,.7);
      }
      [data-testid="stAlert"] {border-radius: 14px;}

      .footer-note {
        margin-top: 1.6rem;
        padding: .9rem 1rem;
        border: 1px solid #D8E5EC;
        border-radius: 16px;
        background: rgba(255,255,255,.72);
        color: #687B87;
        font-size: .78rem;
        line-height: 1.55;
      }
      .footer-note strong {color: #35505E;}

      @media (max-width: 700px) {
        .block-container {padding-top: .65rem; padding-left: .75rem; padding-right: .75rem;}
        .messenger-header {min-height: 82px; padding: .9rem; border-radius: 20px 20px 14px 14px;}
        .profile-avatar {width: 49px; height: 49px; border-radius: 17px;}
        .profile-copy h1 {font-size: 1.18rem;}
        .profile-copy p {font-size: .72rem; white-space: nowrap; overflow: hidden; text-overflow: ellipsis;}
        .header-badge {display: none;}
        .welcome-bubble, [data-testid="stChatMessage"] [data-testid="stChatMessageContent"] {max-width: 88%;}
      }
    </style>
    """,
    unsafe_allow_html=True,
)


def get_secret(name: str, default: str = "") -> str:
    try:
        value = st.secrets.get(name, os.getenv(name, default))
        if not value and name == "OPENAI_API_KEY":
            value = st.secrets.get("model_api_key", os.getenv("model_api_key", default))
        return str(value)
    except FileNotFoundError:
        return os.getenv(name, default)


@st.cache_resource(show_spinner=False)
def build_store(api_key: str, pdf_signature: int):
    del pdf_signature
    client = create_openai_client(api_key)
    pages = load_pdf_pages(Path(__file__).with_name(PDF_NAME))
    chunks = build_chunks(pages)
    vectors = embed_texts(client, [chunk.text for chunk in chunks], DEFAULT_EMBEDDING_MODEL)
    return client, chunks, create_faiss_index(vectors)


def render_sources(hits):
    pages_seen = set()
    with st.expander("근거 문서 / Sources", expanded=False):
        for hit in hits:
            key = hit.chunk.page
            if key in pages_seen:
                continue
            pages_seen.add(key)
            excerpt = hit.chunk.text.replace("\n", " ")
            if len(excerpt) > 280:
                excerpt = excerpt[:277].rstrip() + "..."
            st.markdown(
                f"""
                <div class="source-card">
                  <strong>{hit.chunk.source} · p. {hit.chunk.page}</strong><br/>
                  <small>{excerpt}</small>
                </div>
                """,
                unsafe_allow_html=True,
            )


api_key = get_secret("OPENAI_API_KEY")
chat_model = get_secret("OPENAI_MODEL", DEFAULT_CHAT_MODEL) or model_name()
pdf_path = Path(__file__).with_name(PDF_NAME)

if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    st.markdown(
        """
        <div class="side-profile">
          <div class="side-avatar">💬</div>
          <div>
            <strong>BUFS 보험 도우미</strong>
            <small>Document-grounded assistant</small>
          </div>
        </div>
        """,
        unsafe_allow_html=True,
    )
    st.markdown("#### 채팅 설정")
    language_label = st.radio(
        "답변 언어 / Answer language",
        ("자동 감지 / Auto", "한국어", "English"),
        index=0,
    )
    selected_language = {
        "자동 감지 / Auto": "auto",
        "한국어": "ko",
        "English": "en",
    }[language_label]
    st.markdown(
        f"""
        <div style="padding:.72rem .8rem;border-radius:14px;background:#EAF6FC;
                    color:#41606F;font-size:.78rem;line-height:1.65;">
          <b style="color:#168AC1;">● 문서 연결됨</b><br/>
          Model&nbsp; <code>{chat_model}</code><br/>
          Source&nbsp; <code>{PDF_NAME}</code><br/>
          Pages&nbsp; 7
        </div>
        """,
        unsafe_allow_html=True,
    )
    if st.button("🧹 대화 내용 지우기", use_container_width=True):
        st.session_state.messages = []
        st.rerun()
    st.divider()
    st.warning(SCHOOL_CONFLICT_KO)
    if not api_key:
        st.error(
            "OPENAI_API_KEY가 설정되지 않았습니다. 로컬에서는 환경변수, "
            "배포 환경에서는 Streamlit Secrets에 등록하세요."
        )

st.markdown(
    """
    <div class="messenger-header">
      <div class="profile-avatar">🏥<span class="online-dot"></span></div>
      <div class="profile-copy">
        <h1>외국인 유학생 건강보험 도우미</h1>
        <p>BUFS International Student Support · 지금 상담 가능</p>
      </div>
      <div class="header-badge">PDF RAG</div>
    </div>
    <div class="status-strip">
      <span class="status-pill">🟢 Online</span>
      <span class="status-pill">📄 PDF 7 pages</span>
      <span class="status-pill">🌐 한국어 · English</span>
      <span class="status-pill">🔒 문서 근거 답변</span>
    </div>
    """,
    unsafe_allow_html=True,
)

if not st.session_state.messages:
    st.markdown(
        """
        <div class="welcome-row">
          <div class="welcome-avatar">AI</div>
          <div class="welcome-bubble">
            안녕하세요! <strong>BUFS 건강보험 도우미</strong>예요. 👋<br/>
            보험 가입, 보험료 납부, 병원 이용, 자격확인서 발급을 물어보세요.
            <small>Health_Insurance_info.pdf 안의 내용만 근거로 답변합니다.</small>
          </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

st.markdown(
    """
    <div class="quick-label">
      <strong>빠른 질문</strong>
      <span>버튼을 누르면 바로 질문해요</span>
    </div>
    """,
    unsafe_allow_html=True,
)
example_questions = [
    ("💳 보험료 납부 방법", "국민건강보험료는 어떻게 납부하나요?"),
    ("🏥 병원에서 이용하기", "병원에서 건강보험을 어떻게 사용하나요?"),
    ("📄 자격확인서 발급", "자격득실확인서는 어떻게 발급받나요?"),
    ("🌐 NHIS certificate", "How can I get an NHIS certificate?"),
]
cols = st.columns(2)
selected_example = None
for index, (label, example) in enumerate(example_questions):
    if cols[index % 2].button(label, use_container_width=True, key=f"example-{index}"):
        selected_example = example

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        if message.get("notice"):
            st.info(message["notice"])
        if message.get("hits"):
            render_sources(message["hits"])

typed_question = st.chat_input(
    "건강보험에 대해 질문하세요 / Ask about health insurance",
    disabled=not api_key or not pdf_path.exists(),
)
question = selected_example or typed_question

if question:
    language = detect_language(question, selected_language)
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        try:
            with st.spinner("PDF 근거를 찾고 있습니다... / Searching the PDF..."):
                signature = pdf_path.stat().st_mtime_ns
                client, chunks, index = build_store(api_key, signature)
                hits = retrieve(
                    question,
                    client,
                    index,
                    chunks,
                    language=language,
                    top_k=5,
                )
                answer = answer_question(
                    question,
                    hits,
                    client,
                    language=language,
                    model=chat_model,
                )

            if not hits:
                answer = UNSUPPORTED_KO if language == "ko" else UNSUPPORTED_EN
            st.markdown(answer)
            notice = None
            if should_show_change_notice(answer, language):
                notice = CHANGE_NOTICE_KO if language == "ko" else CHANGE_NOTICE_EN
                st.info(notice)
            if answer not in {UNSUPPORTED_KO, UNSUPPORTED_EN}:
                hits = extract_cited_pages(answer, hits)
                render_sources(hits)
            st.session_state.messages.append(
                {
                    "role": "assistant",
                    "content": answer,
                    "notice": notice,
                    "hits": hits if answer not in {UNSUPPORTED_KO, UNSUPPORTED_EN} else [],
                }
            )
        except Exception as exc:
            st.error(
                "답변을 생성하지 못했습니다. API 키, 모델 접근 권한, 네트워크 상태를 확인하세요. "
                f"오류 유형: {type(exc).__name__}"
            )

st.markdown(
    f"""
    <div class="footer-note">
      <strong>안내</strong> · 이 서비스는 행정·의료 상담을 대체하지 않습니다.
      보험료와 제도는 변경될 수 있으므로 국민건강보험공단에 재확인하세요.<br/>
      <span>{SCHOOL_CONFLICT_EN}</span>
    </div>
    """,
    unsafe_allow_html=True,
)

```

## 10. 배포

1. `app.py`, `rag_core.py`, `Health_Insurance_info.pdf`, `requirements.txt`를 GitHub 공개 저장소에 올린다.
2. Streamlit Community Cloud에서 `app.py`를 엔트리 포인트로 선택한다.
3. Secrets에 `OPENAI_API_KEY`와 `OPENAI_MODEL = "gpt-5.6-luna"`를 등록한다.
4. 모바일·비로그인 환경에서 한국어/영어 질문과 페이지 출처를 확인한다.

보험료와 제도는 변경될 수 있다. 최신 정보는 NHIS(033-811-2000)에 재확인해야 한다.